In [7]:
# =====================================
# ENRIQUECIMIENTO CUALITATIVO CON LLM (Gemini)
# =====================================

import os
import json
import time
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# Cargar la API key desde el fichero .env
load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    print("❌ No se encontró GEMINI_API_KEY en el fichero .env")
else:
    print(f"✅ API key cargada (termina en ...{API_KEY[-4:]})")
    client = genai.Client(api_key=API_KEY)

# Rutas
GOLD_DIR = Path("../data/gold")
PROCESSED_DIR = Path("../data/processed")

✅ API key cargada (termina en ...3isw)


In [6]:
from pathlib import Path

# ¿Existe el fichero .env?
env_path = Path("../.env")
print(f"Ruta buscada: {env_path.resolve()}")
print(f"¿Existe?: {env_path.exists()}")

if env_path.exists():
    with open(env_path) as f:
        contenido = f.read()
    print(f"\nNúmero de líneas: {len(contenido.splitlines())}")
    # Mostrar solo la estructura, no la clave
    for i, linea in enumerate(contenido.splitlines(), 1):
        if '=' in linea:
            nombre = linea.split('=')[0]
            valor = linea.split('=', 1)[1]
            print(f"  Línea {i}: variable '{nombre}' con valor de {len(valor)} caracteres")
        else:
            print(f"  Línea {i}: (sin '=') → '{linea[:30]}...'")

Ruta buscada: /Users/juana/Desktop/tfm-data-science-gtm/.env
¿Existe?: True

Número de líneas: 2
  Línea 1: (sin '=') → '...'
  Línea 2: (sin '=') → 'GEMINI_API_KEYAQ.Ab8RN6LA_eHK-...'


In [8]:
# =====================================
# PRUEBA DE CONEXIÓN CON GEMINI
# =====================================

try:
    respuesta = client.models.generate_content(
        model="gemini-2.0-flash",
        contents="Responde solo con la palabra: OK"
    )
    print("✅ Conexión funcionando")
    print(f"Respuesta: {respuesta.text}")
except Exception as e:
    print(f"❌ Error de conexión:")
    print(f"   {type(e).__name__}: {e}")

❌ Error de conexión:
   ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'status': 'NOT_FOUND'}}


In [9]:
# =====================================
# PRUEBA DE CONEXIÓN CON GEMINI
# =====================================

MODELO = "gemini-3.6-flash"

try:
    respuesta = client.models.generate_content(
        model=MODELO,
        contents="Responde solo con la palabra: OK"
    )
    print("✅ Conexión funcionando")
    print(f"Modelo: {MODELO}")
    print(f"Respuesta: {respuesta.text}")
except Exception as e:
    print(f"❌ Error de conexión:")
    print(f"   {type(e).__name__}: {e}")

✅ Conexión funcionando
Modelo: gemini-3.6-flash
Respuesta: OK


In [10]:
# =====================================
# CARGAR CAPA GOLD Y TOMAR MUESTRA
# =====================================

gold = pd.read_parquet(GOLD_DIR / "gold_restaurantes_madrid.parquet")
print(f"Capa gold cargada: {len(gold)} restaurantes")

# Tomar una muestra de 5 restaurantes CON datos ricos para la prueba
# (que tengan cocina y barrio, para que el LLM tenga con qué trabajar)
muestra = gold[gold['cocina'].notna() & gold['barrio'].notna()].head(5)

print(f"\n=== MUESTRA DE PRUEBA ===")
print(muestra[['nombre', 'barrio', 'cocina', 'epigrafe_oficial', 'tiene_web']].to_string())

Capa gold cargada: 1688 restaurantes

=== MUESTRA DE PRUEBA ===
               nombre                barrio         cocina epigrafe_oficial  tiene_web
0  La Casa del Abuelo  SOL                        regional  BAR RESTAURANTE       True
1             Barinka  SOL                        peruvian      RESTAURANTE      False
2      Taberna Griega  UNIVERSIDAD                   greek      RESTAURANTE       True
5         Maricastaña  UNIVERSIDAD           international  BAR RESTAURANTE       True
6           La Prensa  UNIVERSIDAD                regional  BAR RESTAURANTE      False


In [11]:
# =====================================
# PROMPT Y FUNCIÓN DE ENRIQUECIMIENTO
# =====================================

def construir_prompt(restaurante):
    """Construye el prompt para un restaurante concreto."""
    nombre = restaurante['nombre']
    barrio = restaurante['barrio'].strip() if pd.notna(restaurante['barrio']) else "desconocido"
    cocina = restaurante['cocina'] if pd.notna(restaurante['cocina']) else "no especificada"
    epigrafe = restaurante['epigrafe_oficial'] if pd.notna(restaurante['epigrafe_oficial']) else "no especificado"
    tiene_web = "sí" if restaurante['tiene_web'] else "no"

    prompt = f"""Eres un analista de mercado especializado en el sector de restauración en Madrid.

A partir de la información disponible de un restaurante, infiere sus características comerciales. Basa tu inferencia en el nombre, el tipo de cocina y el barrio. No inventes datos factuales concretos (teléfonos, premios, fechas); limítate a caracterizar el perfil comercial.

DATOS DEL RESTAURANTE:
- Nombre: {nombre}
- Barrio (distrito Centro de Madrid): {barrio}
- Tipo de cocina: {cocina}
- Clasificación: {epigrafe}
- Tiene web propia: {tiene_web}

Devuelve ÚNICAMENTE un objeto JSON válido, sin texto adicional ni markdown, con esta estructura exacta:
{{
  "posicionamiento_precio": "uno de: low_cost, medio, premium, alta_gama",
  "tipo_clientela": "uno de: turista, profesional, residente, mixta",
  "ambiente": "uno de: moderno, tradicional, casual, formal, familiar",
  "presencia_digital": "uno de: nula, basica, activa, sofisticada",
  "resumen": "una o dos frases describiendo el perfil comercial del restaurante"
}}"""
    return prompt

# Probar el prompt con el primer restaurante
prompt_ejemplo = construir_prompt(muestra.iloc[0])
print(prompt_ejemplo)

Eres un analista de mercado especializado en el sector de restauración en Madrid.

A partir de la información disponible de un restaurante, infiere sus características comerciales. Basa tu inferencia en el nombre, el tipo de cocina y el barrio. No inventes datos factuales concretos (teléfonos, premios, fechas); limítate a caracterizar el perfil comercial.

DATOS DEL RESTAURANTE:
- Nombre: La Casa del Abuelo
- Barrio (distrito Centro de Madrid): SOL
- Tipo de cocina: regional
- Clasificación: BAR RESTAURANTE
- Tiene web propia: sí

Devuelve ÚNICAMENTE un objeto JSON válido, sin texto adicional ni markdown, con esta estructura exacta:
{
  "posicionamiento_precio": "uno de: low_cost, medio, premium, alta_gama",
  "tipo_clientela": "uno de: turista, profesional, residente, mixta",
  "ambiente": "uno de: moderno, tradicional, casual, formal, familiar",
  "presencia_digital": "uno de: nula, basica, activa, sofisticada",
  "resumen": "una o dos frases describiendo el perfil comercial del rest

In [12]:
# =====================================
# PROBAR ENRIQUECIMIENTO CON LA MUESTRA
# =====================================

def enriquecer_restaurante(restaurante):
    """Envía un restaurante a Gemini y devuelve el JSON parseado."""
    prompt = construir_prompt(restaurante)
    try:
        respuesta = client.models.generate_content(
            model=MODELO,
            contents=prompt
        )
        texto = respuesta.text.strip()
        # Limpiar posibles marcas de markdown (```json ... ```)
        texto = texto.replace("```json", "").replace("```", "").strip()
        return json.loads(texto)
    except json.JSONDecodeError:
        return {"error": "respuesta no es JSON válido", "raw": texto[:200]}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}

# Probar con los 5 de la muestra
print("=== PRUEBA DE ENRIQUECIMIENTO ===\n")
for idx, row in muestra.iterrows():
    print(f"🍽️  {row['nombre']} ({row['cocina']}, {row['barrio'].strip()})")
    resultado = enriquecer_restaurante(row)
    for clave, valor in resultado.items():
        print(f"    {clave}: {valor}")
    print()
    time.sleep(2)  # pausa para no saturar la API

=== PRUEBA DE ENRIQUECIMIENTO ===

🍽️  La Casa del Abuelo (regional, SOL)


    posicionamiento_precio: medio
    tipo_clientela: turista
    ambiente: tradicional
    presencia_digital: activa
    resumen: Establecimiento tradicional de cocina regional ubicado en un enclave estratégico de alto tránsito en el centro de Madrid. Su perfil comercial se enfoca en captar al público visitante y turístico mediante una propuesta castiza respaldada por canal digital propio.

🍽️  Barinka (peruvian, SOL)
    error: ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

🍽️  Taberna Griega (greek, UNIVERSIDAD)


    posicionamiento_precio: medio
    tipo_clientela: mixta
    ambiente: casual
    presencia_digital: activa
    resumen: Restaurante de cocina griega informal ubicado en el dinámico barrio de Universidad (Malasaña), orientado a un público variado que incluye residentes, estudiantes y visitantes. Ofrece una propuesta gastronómica mediterránea tradicional y accesible, respaldada por su propia plataforma web.

🍽️  Maricastaña (international, UNIVERSIDAD)
    error: ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

🍽️  La Prensa (regional, UNIVERSIDAD)


    posicionamiento_precio: medio
    tipo_clientela: mixta
    ambiente: tradicional
    presencia_digital: basica
    resumen: Establecimiento de corte tradicional enfocado en la cocina regional, que atiende tanto a residentes del barrio de Universidad como a visitantes y trabajadores del distrito Centro. Su modelo comercial responde al clásico bar-restaurante de proximidad, apoyado principalmente en el tráfico peatonal directo dada su escasa presencia digital.



In [13]:
# =====================================
# FUNCIÓN ROBUSTA CON REINTENTOS Y CACHE
# =====================================

# Directorio de cache: cada restaurante enriquecido se guarda como fichero JSON individual
# Así si el proceso se corta, retomamos desde donde estábamos sin rehacer nada
CACHE_DIR = PROCESSED_DIR / "cache_llm"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def enriquecer_con_reintentos(restaurante, max_reintentos=3, espera_base=5):
    """
    Enriquece un restaurante con Gemini.
    - Reintenta hasta max_reintentos veces si hay error 503 o de red.
    - Usa espera exponencial: 5s, 10s, 20s...
    - Cachea el resultado en disco para no repetir llamadas.
    """
    restaurante_id = restaurante['restaurante_id']
    cache_path = CACHE_DIR / f"{restaurante_id}.json"

    # Si ya está cacheado, devolvemos el resultado guardado
    if cache_path.exists():
        with open(cache_path) as f:
            return json.load(f)

    prompt = construir_prompt(restaurante)

    for intento in range(max_reintentos):
        try:
            respuesta = client.models.generate_content(
                model=MODELO,
                contents=prompt
            )
            texto = respuesta.text.strip()
            texto = texto.replace("```json", "").replace("```", "").strip()
            resultado = json.loads(texto)

            # Guardar en cache
            with open(cache_path, 'w') as f:
                json.dump(resultado, f, ensure_ascii=False, indent=2)
            return resultado

        except json.JSONDecodeError:
            # No merece la pena reintentar si el JSON está mal
            return {"error": "json_invalido", "raw": texto[:200]}

        except Exception as e:
            error_str = str(e)
            # Si es 503 o error temporal, esperamos y reintentamos
            if "503" in error_str or "UNAVAILABLE" in error_str or "429" in error_str:
                if intento < max_reintentos - 1:
                    espera = espera_base * (2 ** intento)
                    print(f"    ⏳ Error temporal, esperando {espera}s (intento {intento+1}/{max_reintentos})")
                    time.sleep(espera)
                    continue
            # Otros errores o último intento
            return {"error": f"{type(e).__name__}: {error_str[:200]}"}

    return {"error": "max_reintentos_alcanzado"}

print("✅ Función de enriquecimiento robusta lista")
print(f"Cache en: {CACHE_DIR}")

✅ Función de enriquecimiento robusta lista
Cache en: ../data/processed/cache_llm


In [14]:
# =====================================
# REEJECUTAR MUESTRA CON FUNCIÓN ROBUSTA
# =====================================

print("=== ENRIQUECIMIENTO CON REINTENTOS Y CACHE ===\n")

for idx, row in muestra.iterrows():
    print(f"🍽️  {row['nombre']} ({row['cocina']}, {row['barrio'].strip()})")
    resultado = enriquecer_con_reintentos(row)
    for clave, valor in resultado.items():
        print(f"    {clave}: {valor}")
    print()
    time.sleep(1)

# Ver cuántos ficheros hay en cache
print(f"\n📦 Total en cache: {len(list(CACHE_DIR.glob('*.json')))} restaurantes")

=== ENRIQUECIMIENTO CON REINTENTOS Y CACHE ===

🍽️  La Casa del Abuelo (regional, SOL)


    posicionamiento_precio: medio
    tipo_clientela: mixta
    ambiente: tradicional
    presencia_digital: activa
    resumen: Establecimiento emblemático enfocado en el tapeo y la gastronomía regional dentro de una zona de altísimo tránsito peatonal en pleno centro de Madrid. Su perfil comercial combina el atractivo histórico para el turismo con el consumo casual de público local en un ambiente castizo.

🍽️  Barinka (peruvian, SOL)


    posicionamiento_precio: medio
    tipo_clientela: mixta
    ambiente: casual
    presencia_digital: basica
    resumen: Propuesta de cocina peruana de perfil casual y precio medio, orientada a captar el alto flujo peatonal del barrio de Sol. Su estrategia comercial se apoya en la popularidad de la gastronomía latina y la rotación de clientes en el centro, prescindiendo de infraestructura web propia.

🍽️  Taberna Griega (greek, UNIVERSIDAD)


    posicionamiento_precio: medio
    tipo_clientela: mixta
    ambiente: casual
    presencia_digital: activa
    resumen: Restaurante de cocina griega accesible ubicado en el céntrico y dinámico barrio de Universidad, enfocado a una clientela variada de residentes, jóvenes y visitantes. Su formato de taberna tradicional se complementa con un canal digital propio para captar demanda en la zona.

🍽️  Maricastaña (international, UNIVERSIDAD)


    posicionamiento_precio: medio
    tipo_clientela: mixta
    ambiente: casual
    presencia_digital: activa
    resumen: Establecimiento de corte cosmopolita situado en el dinámico barrio de Universidad, con un formato híbrido de bar y restaurante adaptado a diferentes momentos del día. Su propuesta internacional e informal atrae a un público diverso compuesto por residentes locales, jóvenes profesionales y turistas.

🍽️  La Prensa (regional, UNIVERSIDAD)


    posicionamiento_precio: medio
    tipo_clientela: mixta
    ambiente: tradicional
    presencia_digital: nula
    resumen: Bar-restaurante de corte tradicional centrado en cocina regional y propuesta accesible, orientado a captar tanto a vecinos como a transeúntes en el céntrico barrio de Universidad. Su modelo de negocio se apoya en el tráfico peatonal y la fidelización local, operando sin infraestructura digital propia.


📦 Total en cache: 5 restaurantes


In [15]:
# =====================================
# ENRIQUECIMIENTO DE MUESTRA (40 restaurantes)
# =====================================

# Muestra aleatoria fija (con seed para reproducibilidad) de 40 restaurantes
# Que tengan al menos nombre y coordenadas
muestra_40 = gold[gold['nombre'].notna()].sample(n=40, random_state=42)

print(f"Muestra: {len(muestra_40)} restaurantes")
print(f"Iniciando enriquecimiento...\n")

inicio = time.time()
resultados = []
errores = 0

for i, (idx, row) in enumerate(muestra_40.iterrows(), 1):
    resultado = enriquecer_con_reintentos(row)
    resultado['restaurante_id'] = row['restaurante_id']
    resultado['nombre'] = row['nombre']
    resultados.append(resultado)
    
    if 'error' in resultado:
        errores += 1
        print(f"  [{i}/40] ❌ {row['nombre']}: {resultado.get('error', '')[:80]}")
    else:
        print(f"  [{i}/40] ✅ {row['nombre']}")
    
    time.sleep(1)

duracion = time.time() - inicio

print(f"\n=== RESUMEN ===")
print(f"Duración total:       {duracion:.1f} segundos ({duracion/60:.1f} minutos)")
print(f"Tiempo por restaurante: {duracion/40:.1f} segundos")
print(f"Éxitos:               {40 - errores} / 40 ({(40-errores)/40*100:.0f}%)")
print(f"Errores:              {errores} / 40")
print(f"\nProyección para 1.688 restaurantes: {duracion*1688/40/60:.0f} minutos")

Muestra: 40 restaurantes
Iniciando enriquecimiento...



  [1/40] ✅ Los Arrieros
    ⏳ Error temporal, esperando 5s (intento 1/3)
    ⏳ Error temporal, esperando 10s (intento 2/3)
  [2/40] ❌ La Tía Cebolla: ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is 
    ⏳ Error temporal, esperando 5s (intento 1/3)


  [3/40] ✅ Saporem Ventura
    ⏳ Error temporal, esperando 5s (intento 1/3)
    ⏳ Error temporal, esperando 10s (intento 2/3)


  [4/40] ✅ Los Ángeles


  [5/40] ✅ Ciento Ochenta° de La TraMoya


  [6/40] ✅ El Tigre del Norte
    ⏳ Error temporal, esperando 5s (intento 1/3)


  [7/40] ✅ Mesón de la Tortilla
    ⏳ Error temporal, esperando 5s (intento 1/3)
    ⏳ Error temporal, esperando 10s (intento 2/3)
  [8/40] ❌ La Fondue de Tell: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exc
    ⏳ Error temporal, esperando 5s (intento 1/3)
    ⏳ Error temporal, esperando 10s (intento 2/3)
  [9/40] ❌ La Nieta: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exc
    ⏳ Error temporal, esperando 5s (intento 1/3)
    ⏳ Error temporal, esperando 10s (intento 2/3)
  [10/40] ❌ El Luarqués: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exc
    ⏳ Error temporal, esperando 5s (intento 1/3)
    ⏳ Error temporal, esperando 10s (intento 2/3)
  [11/40] ❌ Sports & Tapas Bar Madrid: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exc
    ⏳ Error temporal, esperando 5s (intento 1/3)
    ⏳ Error temporal, esperando 10s (intento 2/3)
  [12/40] ❌ Azura Bar: ClientError

In [17]:
MODELO = "gemini-3.5-flash-lite"

try:
    respuesta = client.models.generate_content(
        model=MODELO,
        contents="Responde solo con: OK"
    )
    print(f"✅ {MODELO} funciona")
    print(f"Respuesta: {respuesta.text}")
except Exception as e:
    print(f"❌ Error con {MODELO}:")
    print(f"   {type(e).__name__}: {str(e)[:200]}")

✅ gemini-3.5-flash-lite funciona
Respuesta: OK


In [18]:
# =====================================
# PRUEBA DE CALIDAD DEL MODELO LITE
# =====================================

MODELO = "gemini-3.5-flash-lite"

# Cogemos un restaurante que ya enriquecimos con el modelo grande
# para comparar calidades
prueba = muestra.iloc[0]  # La Casa del Abuelo
print(f"Probando con: {prueba['nombre']}\n")

# Importante: para evitar cache del modelo anterior, hacemos la llamada directa
prompt = construir_prompt(prueba)

try:
    respuesta = client.models.generate_content(
        model=MODELO,
        contents=prompt
    )
    texto = respuesta.text.strip().replace("```json", "").replace("```", "").strip()
    resultado = json.loads(texto)
    
    print("=== RESULTADO CON gemini-3.5-flash-lite ===")
    for clave, valor in resultado.items():
        print(f"  {clave}: {valor}")
except Exception as e:
    print(f"❌ Error: {e}")

Probando con: La Casa del Abuelo



=== RESULTADO CON gemini-3.5-flash-lite ===
  posicionamiento_precio: medio
  tipo_clientela: mixta
  ambiente: tradicional
  presencia_digital: basica
  resumen: Establecimiento castizo de cocina regional ubicado en pleno corazón de Sol, orientado tanto al público local como al turismo masivo que busca una experiencia gastronómica tradicional en Madrid.


In [19]:
# =====================================
# ENRIQUECIMIENTO POR TANDAS - TANDA 1
# =====================================

# Configuración del modelo
MODELO = "gemini-3.5-flash-lite"

# Tamaño de la tanda (dejar margen sobre 500/día)
TAMANO_TANDA = 400
PAUSA_ENTRE_LLAMADAS = 4  # segundos (15 rpm = 4s entre llamadas)

# Restaurantes pendientes (los que no están en cache)
ids_cacheados = {f.stem for f in CACHE_DIR.glob("*.json")}
pendientes = gold[~gold['restaurante_id'].isin(ids_cacheados)]
print(f"Total en cache:      {len(ids_cacheados)}")
print(f"Pendientes:          {len(pendientes)}")

# Coger la tanda de hoy
tanda = pendientes.head(TAMANO_TANDA)
print(f"Tanda de esta sesión: {len(tanda)}\n")

# Enriquecer
inicio = time.time()
exitos = 0
errores = 0

for i, (idx, row) in enumerate(tanda.iterrows(), 1):
    resultado = enriquecer_con_reintentos(row)
    
    if 'error' in resultado:
        errores += 1
        # Solo mostramos los errores para no saturar el log
        print(f"  [{i}/{len(tanda)}] ❌ {row['nombre']}: {resultado.get('error', '')[:80]}")
    else:
        exitos += 1
        # Mostramos avance cada 25 exitosos
        if exitos % 25 == 0:
            print(f"  [{i}/{len(tanda)}] ✅ {exitos} enriquecidos hasta ahora ({row['nombre']})")
    
    time.sleep(PAUSA_ENTRE_LLAMADAS)

duracion = time.time() - inicio

print(f"\n=== RESUMEN DE LA TANDA ===")
print(f"Duración total:    {duracion/60:.1f} minutos")
print(f"Éxitos:            {exitos} / {len(tanda)} ({exitos/len(tanda)*100:.1f}%)")
print(f"Errores:           {errores} / {len(tanda)}")
print(f"Total en cache:    {len(list(CACHE_DIR.glob('*.json')))}")

Total en cache:      0
Pendientes:          1688
Tanda de esta sesión: 400



  [25/400] ✅ 25 enriquecidos hasta ahora (La Chamana)


  [50/400] ✅ 50 enriquecidos hasta ahora (Kuoco)


  [75/400] ✅ 75 enriquecidos hasta ahora (Sultan Palast)


  [100/400] ✅ 100 enriquecidos hasta ahora (Kechua)


...

  [305/400] ✅ 300 enriquecidos hasta ahora (Lamucca de Prado)


  [330/400] ✅ 325 enriquecidos hasta ahora (De María)


  [355/400] ✅ 350 enriquecidos hasta ahora (Fan Hua)


  [380/400] ✅ 375 enriquecidos hasta ahora (La Cantina)



=== RESUMEN DE LA TANDA ===
Duración total:    153.2 minutos
Éxitos:            395 / 400 (98.8%)
Errores:           5 / 400
Total en cache:    395


In [20]:
# Ver cuántos restaurantes hay en cache y muestra de uno
ficheros_cache = list(CACHE_DIR.glob("*.json"))
print(f"Total ficheros en cache: {len(ficheros_cache)}")

# Ver uno de ejemplo
if ficheros_cache:
    with open(ficheros_cache[0]) as f:
        ejemplo = json.load(f)
    print(f"\nEjemplo de fichero ({ficheros_cache[0].name}):")
    for k, v in ejemplo.items():
        print(f"  {k}: {v}")

Total ficheros en cache: 395

Ejemplo de fichero (5d4eb8b85691.json):
  posicionamiento_precio: low_cost
  tipo_clientela: mixta
  ambiente: casual
  presencia_digital: nula
  resumen: Restaurante mexicano de tipo informal ubicado en el dinámico barrio de Universidad, enfocado en un público amplio de residentes y visitantes que buscan comida rápida, accesible y sin pretensiones.


In [22]:
# =====================================
# ANÁLISIS DE LAS RESPUESTAS DEL LLM
# =====================================

# Cargar todos los resultados del cache
resultados_cache = []
for f in CACHE_DIR.glob("*.json"):
    with open(f) as fp:
        r = json.load(fp)
        r['restaurante_id'] = f.stem
        resultados_cache.append(r)

df_llm = pd.DataFrame(resultados_cache)
print(f"Total en cache: {len(df_llm)}")
print(f"Columnas: {list(df_llm.columns)}\n")

# Filtrar solo los que tienen respuesta correcta (todas las claves esperadas)
if 'posicionamiento_precio' in df_llm.columns:
    df_ok = df_llm[df_llm['posicionamiento_precio'].notna()].copy()
    print(f"Respuestas exitosas: {len(df_ok)}\n")

    print("=== DISTRIBUCIÓN DE POSICIONAMIENTO ===")
    print(df_ok['posicionamiento_precio'].value_counts())

    print("\n=== DISTRIBUCIÓN DE CLIENTELA ===")
    print(df_ok['tipo_clientela'].value_counts())

    print("\n=== DISTRIBUCIÓN DE AMBIENTE ===")
    print(df_ok['ambiente'].value_counts())

    print("\n=== DISTRIBUCIÓN DE PRESENCIA DIGITAL ===")
    print(df_ok['presencia_digital'].value_counts())

Total en cache: 395
Columnas: ['posicionamiento_precio', 'tipo_clientela', 'ambiente', 'presencia_digital', 'resumen', 'restaurante_id']

Respuestas exitosas: 395

=== DISTRIBUCIÓN DE POSICIONAMIENTO ===
posicionamiento_precio
medio        349
low_cost      18
premium       16
alta_gama     12
Name: count, dtype: int64

=== DISTRIBUCIÓN DE CLIENTELA ===
tipo_clientela
mixta        332
turista       40
residente     23
Name: count, dtype: int64

=== DISTRIBUCIÓN DE AMBIENTE ===
ambiente
casual         188
tradicional    150
moderno         50
formal           6
familiar         1
Name: count, dtype: int64

=== DISTRIBUCIÓN DE PRESENCIA DIGITAL ===
presencia_digital
nula           156
activa         119
basica         117
sofisticada      3
Name: count, dtype: int64


In [23]:
# Prueba de una sola llamada para ver estado de la cuota
try:
    respuesta = client.models.generate_content(
        model=MODELO,
        contents="Responde solo: OK"
    )
    print(f"✅ Cuota disponible. Respuesta: {respuesta.text}")
except Exception as e:
    error_str = str(e)
    if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
        print("❌ Cuota todavía agotada, espera unas horas más")
    else:
        print(f"Otro error: {error_str[:200]}")

✅ Cuota disponible. Respuesta: OK


In [24]:
# =====================================
# ENRIQUECIMIENTO POR TANDAS - TANDA 2
# =====================================

TAMANO_TANDA = 400
PAUSA_ENTRE_LLAMADAS = 4

# Restaurantes pendientes (los que no están en cache)
ids_cacheados = {f.stem for f in CACHE_DIR.glob("*.json")}
pendientes = gold[~gold['restaurante_id'].isin(ids_cacheados)]
print(f"Total en cache:      {len(ids_cacheados)}")
print(f"Pendientes:          {len(pendientes)}")

# Coger la tanda de hoy
tanda = pendientes.head(TAMANO_TANDA)
print(f"Tanda de esta sesión: {len(tanda)}\n")

# Enriquecer
inicio = time.time()
exitos = 0
errores = 0

for i, (idx, row) in enumerate(tanda.iterrows(), 1):
    resultado = enriquecer_con_reintentos(row)
    
    if 'error' in resultado:
        errores += 1
        print(f"  [{i}/{len(tanda)}] ❌ {row['nombre']}: {resultado.get('error', '')[:80]}")
    else:
        exitos += 1
        if exitos % 25 == 0:
            print(f"  [{i}/{len(tanda)}] ✅ {exitos} enriquecidos hasta ahora")
    
    time.sleep(PAUSA_ENTRE_LLAMADAS)

duracion = time.time() - inicio

print(f"\n=== RESUMEN DE LA TANDA ===")
print(f"Duración total:    {duracion/60:.1f} minutos")
print(f"Éxitos:            {exitos} / {len(tanda)} ({exitos/len(tanda)*100:.1f}%)")
print(f"Errores:           {errores} / {len(tanda)}")
print(f"Total en cache:    {len(list(CACHE_DIR.glob('*.json')))}")

Total en cache:      395
Pendientes:          1293
Tanda de esta sesión: 400



  [25/400] ✅ 25 enriquecidos hasta ahora


  [50/400] ✅ 50 enriquecidos hasta ahora


  [75/400] ✅ 75 enriquecidos hasta ahora


  [100/400] ✅ 100 enriquecidos hasta ahora


...

  [325/400] ✅ 325 enriquecidos hasta ahora


  [350/400] ✅ 350 enriquecidos hasta ahora


  [375/400] ✅ 375 enriquecidos hasta ahora


  [399/400] ❌ La Toscana: ReadError: [Errno 54] Connection reset by peer



=== RESUMEN DE LA TANDA ===
Duración total:    49.2 minutos
Éxitos:            399 / 400 (99.8%)
Errores:           1 / 400
Total en cache:    794


In [25]:
# =====================================
# ANÁLISIS DE LAS RESPUESTAS
# =====================================

# Recargar todos los ficheros del cache
resultados_cache = []
for f in CACHE_DIR.glob("*.json"):
    with open(f) as fp:
        r = json.load(fp)
        r['restaurante_id'] = f.stem
        resultados_cache.append(r)

df_llm = pd.DataFrame(resultados_cache)
df_ok = df_llm[df_llm['posicionamiento_precio'].notna()].copy()

print(f"Total respuestas analizadas: {len(df_ok)}\n")

print("=== POSICIONAMIENTO PRECIO ===")
print(df_ok['posicionamiento_precio'].value_counts(normalize=True).round(3) * 100)

print("\n=== TIPO CLIENTELA ===")
print(df_ok['tipo_clientela'].value_counts(normalize=True).round(3) * 100)

print("\n=== AMBIENTE ===")
print(df_ok['ambiente'].value_counts(normalize=True).round(3) * 100)

print("\n=== PRESENCIA DIGITAL ===")
print(df_ok['presencia_digital'].value_counts(normalize=True).round(3) * 100)

Total respuestas analizadas: 794

=== POSICIONAMIENTO PRECIO ===
posicionamiento_precio
medio        89.9
low_cost      4.7
premium       3.1
alta_gama     2.1
alto          0.1
Name: proportion, dtype: float64

=== TIPO CLIENTELA ===
tipo_clientela
mixta        83.8
turista      10.2
residente     6.0
Name: proportion, dtype: float64

=== AMBIENTE ===
ambiente
casual         50.1
tradicional    36.5
moderno        11.3
formal          1.3
familiar        0.8
Name: proportion, dtype: float64

=== PRESENCIA DIGITAL ===
presencia_digital
nula           43.8
basica         30.6
activa         24.6
sofisticada     1.0
Name: proportion, dtype: float64


In [29]:
# Ver cuántos están enriquecidos
ficheros_cache = list(CACHE_DIR.glob("*.json"))
print(f"Total en cache: {len(ficheros_cache)}")
print(f"Pendientes: {1688 - len(ficheros_cache)}")

Total en cache: 1007
Pendientes: 681


In [30]:
try:
    respuesta = client.models.generate_content(
        model=MODELO,
        contents="Responde solo: OK"
    )
    print(f"✅ Cuota disponible. Respuesta: {respuesta.text}")
except Exception as e:
    if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
        print("❌ Cuota agotada")
    else:
        print(f"Otro error: {str(e)[:200]}")

✅ Cuota disponible. Respuesta: OK


In [31]:
# =====================================
# ENRIQUECIMIENTO POR TANDAS - TANDA 3 (continuación)
# =====================================

TAMANO_TANDA = 250  # ajustado por cuota ya consumida
PAUSA_ENTRE_LLAMADAS = 4

# Restaurantes pendientes (los que no están en cache)
ids_cacheados = {f.stem for f in CACHE_DIR.glob("*.json")}
pendientes = gold[~gold['restaurante_id'].isin(ids_cacheados)]
print(f"Total en cache:      {len(ids_cacheados)}")
print(f"Pendientes:          {len(pendientes)}")

tanda = pendientes.head(TAMANO_TANDA)
print(f"Tanda de esta sesión: {len(tanda)}\n")

inicio = time.time()
exitos = 0
errores = 0

for i, (idx, row) in enumerate(tanda.iterrows(), 1):
    resultado = enriquecer_con_reintentos(row)
    
    if 'error' in resultado:
        errores += 1
        print(f"  [{i}/{len(tanda)}] ❌ {row['nombre']}: {resultado.get('error', '')[:80]}")
    else:
        exitos += 1
        if exitos % 25 == 0:
            print(f"  [{i}/{len(tanda)}] ✅ {exitos} enriquecidos hasta ahora")
    
    time.sleep(PAUSA_ENTRE_LLAMADAS)

duracion = time.time() - inicio

print(f"\n=== RESUMEN DE LA TANDA ===")
print(f"Duración total:    {duracion/60:.1f} minutos")
print(f"Éxitos:            {exitos} / {len(tanda)} ({exitos/len(tanda)*100:.1f}%)")
print(f"Errores:           {errores} / {len(tanda)}")
print(f"Total en cache:    {len(list(CACHE_DIR.glob('*.json')))}")

Total en cache:      1007
Pendientes:          681
Tanda de esta sesión: 250

    ⏳ Error temporal, esperando 5s (intento 1/3)
    ⏳ Error temporal, esperando 10s (intento 2/3)


    ⏳ Error temporal, esperando 5s (intento 1/3)


  [18/250] ❌ Come Prima: ReadError: [Errno 54] Connection reset by peer


  [26/250] ✅ 25 enriquecidos hasta ahora


  [51/250] ✅ 50 enriquecidos hasta ahora


...

  [151/250] ✅ 150 enriquecidos hasta ahora


  [176/250] ✅ 175 enriquecidos hasta ahora


  [201/250] ✅ 200 enriquecidos hasta ahora


  [226/250] ✅ 225 enriquecidos hasta ahora



=== RESUMEN DE LA TANDA ===
Duración total:    72.7 minutos
Éxitos:            249 / 250 (99.6%)
Errores:           1 / 250
Total en cache:    1256


In [32]:
try:
    respuesta = client.models.generate_content(
        model=MODELO,
        contents="Responde solo: OK"
    )
    print(f"✅ Cuota disponible")
except Exception as e:
    if "429" in str(e):
        print("❌ Cuota agotada, mañana")
    else:
        print(f"Error: {str(e)[:150]}")

✅ Cuota disponible


In [ ]:
# =====================================
# TANDA HASTA COMPLETAR
# =====================================

TAMANO_TANDA = 432  # los que quedan
PAUSA_ENTRE_LLAMADAS = 4

ids_cacheados = {f.stem for f in CACHE_DIR.glob("*.json")}
pendientes = gold[~gold['restaurante_id'].isin(ids_cacheados)]
print(f"Pendientes: {len(pendientes)}")

tanda = pendientes.head(TAMANO_TANDA)
print(f"Tanda: {len(tanda)}\n")

inicio = time.time()
exitos = 0
errores = 0

for i, (idx, row) in enumerate(tanda.iterrows(), 1):
    resultado = enriquecer_con_reintentos(row)
    
    if 'error' in resultado:
        errores += 1
        # No imprimir todos los errores para no saturar el log si son muchos
        if errores <= 5 or errores % 50 == 0:
            print(f"  [{i}/{len(tanda)}] ❌ {row['nombre']}: {resultado.get('error', '')[:60]}")
    else:
        exitos += 1
        if exitos % 50 == 0:
            print(f"  [{i}/{len(tanda)}] ✅ {exitos} enriquecidos")
    
    time.sleep(PAUSA_ENTRE_LLAMADAS)

duracion = time.time() - inicio

print(f"\n=== RESUMEN FINAL ===")
print(f"Duración:  {duracion/60:.1f} min")
print(f"Éxitos:    {exitos} / {len(tanda)}")
print(f"Errores:   {errores}")
print(f"Total en cache: {len(list(CACHE_DIR.glob('*.json')))} / 1688")

In [34]:
total = len(list(CACHE_DIR.glob("*.json")))
print(f"Total en cache: {total} / 1688 ({total/1688*100:.1f}%)")
print(f"Pendientes: {1688 - total}")

Total en cache: 1480 / 1688 (87.7%)
Pendientes: 208


In [36]:
# =====================================
# TANDA FINAL - CERRAR ENRIQUECIMIENTO
# =====================================

TAMANO_TANDA = 250  # más que los 208 pendientes, para asegurar
PAUSA_ENTRE_LLAMADAS = 4

ids_cacheados = {f.stem for f in CACHE_DIR.glob("*.json")}
pendientes = gold[~gold['restaurante_id'].isin(ids_cacheados)]
print(f"Pendientes: {len(pendientes)}")

tanda = pendientes.head(TAMANO_TANDA)
print(f"Tanda: {len(tanda)}\n")

inicio = time.time()
exitos = 0
errores = 0

for i, (idx, row) in enumerate(tanda.iterrows(), 1):
    resultado = enriquecer_con_reintentos(row)
    
    if 'error' in resultado:
        errores += 1
        print(f"  [{i}/{len(tanda)}] ❌ {row['nombre']}: {resultado.get('error', '')[:60]}")
    else:
        exitos += 1
        if exitos % 25 == 0:
            print(f"  [{i}/{len(tanda)}] ✅ {exitos} enriquecidos")
    
    time.sleep(PAUSA_ENTRE_LLAMADAS)

duracion = time.time() - inicio

print(f"\n=== RESUMEN FINAL ===")
print(f"Duración:  {duracion/60:.1f} min")
print(f"Éxitos:    {exitos} / {len(tanda)}")
print(f"Errores:   {errores}")
print(f"Total en cache: {len(list(CACHE_DIR.glob('*.json')))} / 1688")

Pendientes: 208
Tanda: 208



    ⏳ Error temporal, esperando 5s (intento 1/3)
    ⏳ Error temporal, esperando 10s (intento 2/3)
  [10/208] ❌ Zest Almagro: ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'messa
    ⏳ Error temporal, esperando 5s (intento 1/3)
    ⏳ Error temporal, esperando 10s (intento 2/3)
  [11/208] ❌ Pointer Madrid: ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'messa


    ⏳ Error temporal, esperando 5s (intento 1/3)


    ⏳ Error temporal, esperando 5s (intento 1/3)


    ⏳ Error temporal, esperando 5s (intento 1/3)


...

  [129/208] ✅ 125 enriquecidos


  [154/208] ✅ 150 enriquecidos


  [179/208] ✅ 175 enriquecidos


  [204/208] ✅ 200 enriquecidos



=== RESUMEN FINAL ===
Duración:  132.4 min
Éxitos:    204 / 208
Errores:   4
Total en cache: 1684 / 1688


In [37]:
# Ver qué restaurantes quedaron pendientes
ids_cacheados = {f.stem for f in CACHE_DIR.glob("*.json")}
pendientes = gold[~gold['restaurante_id'].isin(ids_cacheados)]
print(f"Pendientes: {len(pendientes)}\n")
print(pendientes[['nombre', 'cocina', 'barrio']].to_string())

Pendientes: 4

              nombre    cocina barrio
1489    Zest Almagro      None   None
1490  Pointer Madrid     diner   None
1521        Cuenllas  regional   None
1567            None      None   None
